# MIDORI2 CDS and amino-acid shift

This stage links the exact Mammalia mutation-spectrum intersection for **CO1 → CO3 → Cytb** to MIDORI2 GB272. It preserves the chain `mutation-spectrum Species → MIDORI taxonomy/TaxID → selected accession.version and locus → exact DNA + AA pair → QC → final cohort → observed composition → real-codon opportunities → MutSpec weighting → predicted amino-acid shift`. No species is removed without an audit reason.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import Image, display

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents) if (p / '1init_data' / 'data' / 'MutSpecVertebrates12.csv.gz').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run from the repository root or 6amino_acid_shift directory')
STAGE = ROOT / '6amino_acid_shift'
sys.path.insert(0, str(STAGE))
from amino_acid_shift import BIOPYTHON_AVAILABLE, TARGET_GENES, load_mutation_cohort
from midori_analysis import MIDORI_RELEASE, run_midori_analysis

N_BOOTSTRAP = 2000
RANDOM_SEED = 32
MUTATION_SOURCE = ROOT / '1init_data' / 'data' / 'MutSpecVertebrates12.csv.gz'
print(f'MIDORI2 release: {MIDORI_RELEASE}')
print(f'Biopython available for NCBI Taxonomy resolution: {BIOPYTHON_AVAILABLE}')
print(f'NCBI email configured: {bool(os.getenv("NCBI_EMAIL") or os.getenv("ENTREZ_EMAIL"))}')
print(f'NCBI API key configured: {bool(os.getenv("NCBI_API_KEY") or os.getenv("ENTREZ_API_KEY"))}')

## 1. Mutation-spectrum species intersection and exact taxonomy

Species identifiers are matched only after replacing repository underscores with spaces and normalizing whitespace/case. Taxonomic tokens (subspecies, strain, `cf.`, `aff.`, `sp.`) are never discarded. The source species TaxID is parsed from `info.csv` and resolved through NCBI Taxonomy so merged/renamed taxa can be accepted explicitly as `taxid_confirmed`, never by fuzzy matching. MIDORI2 already supplies separate marker databases, so no gene-name substring matching is needed.

In [ ]:
spectra, mutation_availability, mutation_common = load_mutation_cohort(MUTATION_SOURCE, class_filter='Mammalia')
gene_counts = (spectra[['Gene', 'Species']].drop_duplicates().groupby('Gene').size().reindex(TARGET_GENES).rename('n_species').reset_index())
N_CO1, N_CO3, N_Cytb = gene_counts.set_index('Gene')['n_species'].reindex(TARGET_GENES).astype(int)
N_mutation_common = len(mutation_common)
display(gene_counts)
print(f'N_CO1={N_CO1}; N_CO3={N_CO3}; N_Cytb={N_Cytb}; N_mutation_common={N_mutation_common}')
display(mutation_common.head())

## 2. Cached MIDORI2 files, deterministic selection, and QC

The analysis reads cached MIDORI2 GB272 `UNIQ_NUC` and `TOTAL_AA` RAW FASTA files under `data/midori/`. For every nucleotide locus, protein is accepted only when the AA identifier contains the same accession and coordinates. Selection first prefers one accession with basic-valid pairs for all three genes, then uses a deterministic per-gene fallback. All candidates and file SHA-256 hashes are retained. Publication date is not used as a quality criterion.

QC checks coding orientation, vertebrate mitochondrial translation table 2, partialness, ambiguity, internal stops, frame, gene-relative length, and exact equality between translation of the MIDORI nucleotide locus and its paired MIDORI protein. Terminal mitochondrial `T`/`TA` incomplete stops are handled explicitly.

In [ ]:
results = run_midori_analysis(
    MUTATION_SOURCE, STAGE, class_filter='Mammalia',
    n_boot=N_BOOTSTRAP, seed=RANDOM_SEED,
)
display(results['attrition'])

## 3. Species availability and programmatic Results statement

In [ ]:
availability = results['availability']
N_with_CO1_sequence = int(availability['has_seq_CO1'].sum())
N_with_CO3_sequence = int(availability['has_seq_CO3'].sum())
N_with_Cytb_sequence = int(availability['has_seq_Cytb'].sum())
N_with_all_three_sequences = int(availability[['has_seq_CO1', 'has_seq_CO3', 'has_seq_Cytb']].all(axis=1).sum())
N_final_common = int(availability['included_final'].sum())
print(
    f'Mutation-spectrum data were available for all three genes in {N_mutation_common} species.\n'
    f'MIDORI2 sequences for COX1, COX3 and CytB were successfully obtained for {N_with_all_three_sequences} species.\n'
    f'After sequence QC, {N_final_common} species remained in the final matched cohort.'
)
print(f'Per-gene sequence availability: COX1={N_with_CO1_sequence}, COX3={N_with_CO3_sequence}, CytB={N_with_Cytb_sequence}')

In [ ]:
display(results['exclusion_summary'])
display(results['matches']['match_status'].value_counts().rename('n_species').reset_index())
display(results['matches'].loc[results['matches']['match_status'].eq('taxid_confirmed')])
display(results['record_consistency']['same_accession_all_three'].value_counts().rename('n_species').reset_index())
display(results['qc'][['Species', 'Gene', 'accession_version', 'cds_length', 'protein_length', 'terminal_incomplete_stop', 'translation_match', 'sequence_qc_status', 'sequence_qc_reason']].head(12))

## 4. Why protein and CDS answer different questions

The paired MIDORI2 protein supplies actual counts and fractions of the 20 amino acids and the A-rich/C-rich composition scores. The nucleotide CDS supplies the actual codons and codon positions needed to enumerate `A_H>G_H` and `C_H>T_H` opportunities and their synonymous, missense, nonsense, or stop-loss effects. The genetic code is degenerate: lysine, for example, may be `AAA` or `AAG`, so an amino-acid sequence cannot reconstruct the nucleotide opportunities.

For each residue, `A_richness_12` and `C_richness_12` are the mean A or C fraction across its synonymous codons under vertebrate mitochondrial table 2. Their protein-frequency-weighted scores describe observed composition, not theoretical mutation pressure.

In [ ]:
if N_final_common:
    display(results['score_summary'])
    display(results['score_contrasts'])
    display(results['composition_summary'].pivot(index='amino_acid', columns='Gene', values='mean'))
    display(results['opportunities'].groupby(['Gene', 'mutation_H', 'consequence']).size().rename('n_opportunities').reset_index())
    display(results['predicted_summary'].sort_values('mean', key=abs, ascending=False).head(15))

## 5. Observed versus mutation-spectrum-implied gene changes

All summaries resample species as matched clusters. The association across amino acids is exploratory and does not establish causality: selection, phylogeny, gene identity, base composition, and the gene-order/DssH proxy remain confounded.

In [ ]:
if N_final_common:
    display(results['associations'])
    display(Image(filename=str(STAGE / 'figures' / 'midori' / 'main_amino_acid_shift.png')))